In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, QuantileTransformer,KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from xgboost import XGBRegressor
from collections import defaultdict
from tabulate import tabulate
import seaborn as sns
from functools import reduce
import operator
from scipy.stats import uniform, randint
import joblib
import os
from datetime import datetime
import sys
import warnings
import re
import math



RAW_DATA_PATH = "raw.parquet"
PROCESSED_DATA_PATH = "processed.parquet"
MODEL_DIR = "models"
PLOT_DIR = "plots"
ONE_HOT_ENCODING = 'one_hot'
DUMMY_ENCODING = 'dummy'
FEATURE_VARS = ["operation_type", "data_size", "number_of_operations"]
TARGET_VARS = ['latency']
# CATEGORICAL_VARIABLES = {'compaction_style', 'bloom_filter_policy', 'operation_type'}
CATEGORICAL_VARIABLES = {'operation_type'}
MODEL_NAMES = {'mlp', 'xgb'}
AGGREGATE_MEAN, AGGREGATE_MEDIAN, AGGREGATE_NONE = 'AGGREGATE_MEAN', 'AGGREGATE_MEDIAN', 'AGGREGATE_NONE'
HYPERPARAMETER_SAMPLE_FRAC=0.001


# Data Analysis

In [8]:
if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)
    print(f"Created directory: {PLOT_DIR}")
df = pd.read_parquet(RAW_DATA_PATH)
print(f"Loaded DataFrame from: {RAW_DATA_PATH}")

### Validate values ###

for feature in FEATURE_VARS + TARGET_VARS:
    print(f"--- {feature.capitalize()} ---")
    if pd.api.types.is_numeric_dtype(df[feature]):
        print(df[feature].describe())
        print(f"Unique values: {df[feature].nunique()}")
    else:
        print(df[feature].value_counts())

for feature in FEATURE_VARS + TARGET_VARS:
    if pd.api.types.is_numeric_dtype(df[feature]):
        plt.figure(figsize=(8, 6))
        sns.boxplot(x=df[feature])
        plt.title(f"Box Plot of {feature.capitalize()}")
        filepath = os.path.join(PLOT_DIR, f"{feature}_boxplot.png")
        plt.savefig(filepath)
        print(f"Saved box plot: {filepath}") # Log plot saving
        plt.close()

for feature in FEATURE_VARS + TARGET_VARS:
    if not pd.api.types.is_numeric_dtype(df[feature]):
        plt.figure(figsize=(8, 6))
        sns.countplot(x=df[feature])
        plt.title(f"Count of {feature.capitalize()}")
        filepath = os.path.join(PLOT_DIR, f"{feature}_countplot.png")
        plt.savefig(filepath)
        print(f"Saved count plot: {filepath}") # Log plot saving
        plt.close()

Loaded DataFrame from: raw.parquet
--- Operation_type ---
operation_type
GET     16383500
PUT     16383500
SEEK    16383500
Name: count, dtype: int64
--- Data_size ---
count    4.915050e+07
mean     2.880888e+09
std      3.627535e+09
min      1.048576e+08
25%      2.097152e+08
50%      1.073742e+09
75%      5.368709e+09
max      1.073742e+10
Name: data_size, dtype: float64
Unique values: 7
--- Number_of_operations ---
count    4.915050e+07
mean     3.641000e+05
std      1.205139e+05
min      1.000000e+02
25%      4.096000e+05
50%      4.096000e+05
75%      4.096000e+05
max      4.096000e+05
Name: number_of_operations, dtype: float64
Unique values: 5
--- Latency ---
count    4.915050e+07
mean     2.046121e+01
std      1.275713e+04
min      0.000000e+00
25%      5.000000e+00
50%      6.000000e+00
75%      8.000000e+00
max      6.577435e+07
Name: latency, dtype: float64
Unique values: 8311
Saved box plot: plots\data_size_boxplot.png
Saved box plot: plots\number_of_operations_boxplot.png
S

In [16]:
# Iterate through each operation type
for op_type in df['operation_type'].unique():
    op_df = df[df['operation_type'] == op_type]

    # 1. Facet Grid
    g = sns.FacetGrid(op_df, col="number_of_operations", height=5, aspect=1)
    g.map(plt.scatter, "data_size", "latency")
    g.set_axis_labels("Data Size", "Latency")
    g.add_legend()
    filepath = os.path.join(PLOT_DIR, f"{op_type}_facet_grid_data_size.png")
    plt.savefig(filepath)
    plt.close()
    print(f"Saved facet grid: {filepath}")

    # 2. Facet Grid (number_of_operations vs. latency)
    g2 = sns.FacetGrid(op_df, col="data_size", height=5, aspect=1)
    g2.map(plt.scatter, "number_of_operations", "latency")
    g2.set_axis_labels("Number of Operations", "Latency")
    g2.add_legend()
    filepath2 = os.path.join(PLOT_DIR, f"{op_type}_facet_grid_num_ops.png")
    plt.savefig(filepath2)
    plt.close()
    print(f"Saved facet grid (num_ops): {filepath2}")

    # 3. 3D Scatter Plot
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(op_df['data_size'], op_df['number_of_operations'], op_df['latency'], c=op_df.index)
    ax.set_xlabel('Data Size')
    ax.set_ylabel('Number of Operations')
    ax.set_zlabel('Latency')
    plt.title(f"3D Scatter Plot: {op_type}")
    filepath = os.path.join(PLOT_DIR, f"{op_type}_3d_scatter.png")
    plt.savefig(filepath)
    plt.close()
    print(f"Saved 3D scatter plot: {filepath}")

Saved facet grid: plots\GET_facet_grid.png
Saved facet grid (num_ops): plots\GET_facet_grid_num_ops.png
Saved 3D scatter plot: plots\GET_3d_scatter.png
Saved facet grid: plots\PUT_facet_grid.png
Saved facet grid (num_ops): plots\PUT_facet_grid_num_ops.png
Saved 3D scatter plot: plots\PUT_3d_scatter.png
Saved facet grid: plots\SEEK_facet_grid.png
Saved facet grid (num_ops): plots\SEEK_facet_grid_num_ops.png
Saved 3D scatter plot: plots\SEEK_3d_scatter.png


# Domain Objects

In [2]:
MLP_HYPERPARAMETER_SET = {'hidden_layer_sizes', 'activation', 'solver', 'alpha', 'learning_rate_init', 'learning_rate'}
XGB_HYPERPARAMETER_SET = { "n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree"}
HYPERPARAMETER_SPEEDUP_BY_SAMPLING, HYPERPARAMETER_SPEEDUP_USE_AGG= 'HYPERPARAMETER_SPEEDUP_BY_SAMPLING', 'HYPERPARAMETER_SPEEDUP_USE_AGG'
OUTLIER_REMOVAL_STRATEGY_Z_SCORE, OUTLIER_REMOVAL_STRATEGY_ROBUST_Z_SCORE, OUTLIER_REMOVAL_STRATEGY_IQR, OUTLIER_REMOVAL_STRATEGY_Z_SCORE_WINSORIZE = 'OUTLIER_REMOVAL_STRATEGY_Z_SCORE', 'OUTLIER_REMOVAL_STRATEGY_ROBUST_Z_SCORE', 'OUTLIER_REMOVAL_STRATEGY_IQR', 'OUTLIER_REMOVAL_STRATEGY_Z_SCORE_WINSORIZE'

class ModelConfig:
    """
        Configuration class for training and evaluating machine learning models.
    """
    def __init__(self, name: str, encoding_type: str, outlier_removal_strategy: str, feature_vars: list[str], target_vars: list[str], aggregate: str, hyperparameter_speedup_strategy: str = None):
        """
        Initializes a ModelConfig object.

        Args:
            name: Name of the model ('mlp' or 'xgb').
            encoding_type: Encoding type ('one_hot' or 'dummy').
            outlier_removal_strategy: Method to remove outliers
            feature_vars: List of feature variable names.
            target_vars: List of target variable names.
            aggregate: Aggregation type ('mean', 'median', 'none').
        """
        self.name = name
        self.encoding_type = encoding_type
        self.outlier_removal_strategy = outlier_removal_strategy
        self.feature_vars = feature_vars
        self.target_vars = target_vars
        self.aggregate = aggregate
        self.hyperparameter_speedup_strategy = hyperparameter_speedup_strategy

        if self.name == 'mlp':
            self.hyperparameter_set = MLP_HYPERPARAMETER_SET
            self.param_distributions = {
                'hidden_layer_sizes': [(randint(10, 150).rvs(),), (randint(10, 150).rvs(), randint(10, 150).rvs())], #randomly selects 1 or 2 hidden layers, with random sizes.
                'activation': ['relu', 'tanh', 'logistic'],
                'solver': ['adam', 'lbfgs', 'sgd'],
                'alpha': uniform(0.0001, 0.05-0.0001),
                'learning_rate_init': uniform(0.001, 0.1-0.001),
                'learning_rate': ['constant', 'invscaling', 'adaptive']
            }
        elif self.name == 'xgb':
            self.hyperparameter_set = XGB_HYPERPARAMETER_SET
            self.param_distributions = {
                "n_estimators": randint(100, 500),
                "max_depth": randint(3, 10),
                "learning_rate": uniform(0.01, 0.2 - 0.01),
                "subsample": uniform(0.7, 1.0 - 0.7),
                "colsample_bytree": uniform(0.7, 1.0 - 0.7),
            }

        self.best_hyperparameters = defaultdict(lambda: None)

    def clip_param_grid(self, param_grid):
        """
        Clips the parameter grid values to specified ranges.

        Args:
            param_grid: Parameter grid dictionary.

        Returns:
            Clipped parameter grid dictionary.
        """
        clipped_grid = param_grid.copy()
        if self.name == 'mlp':
            clipped_grid["alpha"] = np.clip(clipped_grid["alpha"], 0.0001, 0.05)
            clipped_grid["learning_rate_init"] = np.clip(clipped_grid["learning_rate_init"], 0.001, 0.1)
        elif self.name == 'xgb':
            clipped_grid["learning_rate"] = np.clip(clipped_grid["learning_rate"], 0.01, 0.2)
            clipped_grid["subsample"] = np.clip(clipped_grid["subsample"], 0.7, 1.0)
            clipped_grid["colsample_bytree"] = np.clip(clipped_grid["colsample_bytree"], 0.7, 1.0)
        return clipped_grid

    def __hash__(self):
        return hash((self.name, self.encoding_type, self.outlier_removal_strategy, tuple(self.feature_vars), tuple(self.target_vars), self.aggregate))

    def __eq__(self, other):
        return (
            isinstance(other, ModelConfig)
            and self.name == other.name
            and self.encoding_type == other.encoding_type
            and self.outlier_removal_strategy == other.outlier_removal_strategy
            and self.feature_vars == other.feature_vars
            and self.target_vars == other.target_vars
            and self.aggregate == other.aggregate
        )

    def __repr__(self):
        return f"ModelConfig(name={self.name}, encoding_type={self.encoding_type}, outlier_removal_strategy={self.outlier_removal_strategy}, feature_vars={self.feature_vars}, target_vars={self.target_vars}, agg={self.aggregate}), hyper_param_speedup={self.hyperparameter_speedup_strategy}"


class ModelResult:
    def __init__(self, model: XGBRegressor | MLPRegressor, mse: float, r2: float, best_param_grid: dict):
        self.model = model
        self.mse = mse
        self.r2 = r2
        self.best_param_grid = best_param_grid

    def __repr__(self):
        return f"ModelResult(model={self.model}, mse={self.mse}, r2={self.r2})"


class ModelMap:
    """
    Manages and stores the results of trained models.
    """
    def __init__(self):
        # Store results as: {op_type -> {model_name -> [(ModelConfig, ModelResult)]}}
        self.data: dict[str, dict[str, list[tuple[ModelConfig, ModelResult]]]] = defaultdict(lambda: defaultdict(list))
        self.overall_results: dict[str, list[tuple[ModelConfig, ModelResult]]] = defaultdict(list)  # Stores overall results per model_name

    def addIndivModelResult(self, op_type: str, model_config: ModelConfig, model_result: ModelResult):
        print(f"Added Indiv Result")
        print(self)
        self.data[op_type][model_config.name].append((model_config, model_result))

    def addOverallModelResult(self, model_config: ModelConfig, model_result: ModelResult):
        self.overall_results[model_config.name].append((model_config, model_result))
        print(f"Added Overall Result")
        print(self)
    
    def getModelResult(self, op_type: str, model_config: ModelConfig):
        if op_type == 'OVERALL':
            for data in self.overall_results[model_config.name]:
                conf, result = data
                if conf.aggregate == model_config.aggregate:
                    return result
        else:
            for data in self.data[op_type][model_config.name]:
                conf, result = data
                if conf.aggregate == model_config.aggregate:
                    return result

    def __repr__(self):
        repr_str = "ModelMap:\n"
        for model_name in {name for op in self.data.values() for name in op} | set(self.overall_results):
            repr_str += f"\n=== Model: {model_name} ===\n"
            table_data = []
            
            # Add individual operation type results
            for op_type, models in self.data.items():
                if model_name in models:
                    for config, result in models[model_name]:
                        table_data.append([op_type, config.outlier_removal_strategy, config.encoding_type, config.aggregate, result.mse, result.r2])

            # Add overall results
            for config, result in self.overall_results.get(model_name, []):
                table_data.append(["Overall", config.outlier_removal_strategy, config.encoding_type, config.aggregate, result.mse, result.r2])

            repr_str += tabulate(table_data, headers=["Op Type", "Clean Data", "Encoding Type", "Aggregate", "MSE", "R²"], tablefmt="grid")
            repr_str += "\n"

        return repr_str


def generate_configs(model_name: str):
    """
    Generates an exhaustive list of ModelConfig objects for different configurations.

    Args:
        model_name: Name of the model ('mlp' or 'xgb').

    Returns:
        List of ModelConfig objects.
    """
    res = []
    for encoding_type in [ONE_HOT_ENCODING, DUMMY_ENCODING]:
        for i in range(2):
            for agg in [AGGREGATE_MEAN, AGGREGATE_MEDIAN, AGGREGATE_NONE]:
                for outlier_removal in [OUTLIER_REMOVAL_STRATEGY_Z_SCORE, OUTLIER_REMOVAL_STRATEGY_ROBUST_Z_SCORE, OUTLIER_REMOVAL_STRATEGY_IQR, OUTLIER_REMOVAL_STRATEGY_Z_SCORE_WINSORIZE]:
                    res.append(
                        ModelConfig(
                            model_name,
                            encoding_type,
                            outlier_removal_strategy=outlier_removal,
                            feature_vars=FEATURE_VARS,
                            target_vars=TARGET_VARS,
                            aggregate=agg
                        )
                    )
    return res

def get_adjustment_factor(best_param_value):
    """
    Determines an appropriate adjustment factor for fine-tuning a hyperparameter.

    Args:
        best_param_value: The best hyperparameter value.

    Returns:
        The adjustment factor (e.g., for use in np.linspace), or None if 
        the input is not a float or integer.
    """
    if not isinstance(best_param_value, (float, int)):
        return None  # Ignore non-numeric types

    # Handle zero to avoid log10 errors
    if best_param_value == 0:
      return 0.01

    # Determine the order of magnitude of the best parameter value.
    order_of_magnitude = 10 ** int(np.floor(np.log10(abs(best_param_value))))

    # Adjust the range based on the order of magnitude.
    if order_of_magnitude >= 1:
        adjustment_factor = 0.1 * order_of_magnitude  # 10% for larger values
    elif order_of_magnitude >= 0.01:
        adjustment_factor = 0.01  # 0.01 for values in the 0.01 to 1 range
    else:
        adjustment_factor = 0.005 # 0.005 for values smaller than 0.01

    return adjustment_factor

def get_base_param_grid(best_params: dict, hyper_parameter_set):
    """
        Generates a base parameter grid for GridSearchCV based on RandomizedSearchCV.best_params_ or GridSearchCV.best_params_ .

        Args:
            best_params: RandomizedSearchCV.best_params_ or GridSearchCV.best_params_
            hyper_parameter_set: Set of hyperparameters to include in the grid.

        Returns:
            A dictionary representing the base parameter grid.
    """
    param_grid = {}
    for param in hyper_parameter_set:
        value = best_params[param]
        if isinstance(value, (float)):
            adjustment_factor = get_adjustment_factor(value)
            value = np.linspace(value-adjustment_factor, value+adjustment_factor, 5)
        else:
            value = [value]
        param_grid[param] = value

    return param_grid

def get_param_grid(model_config: ModelConfig, best_params: dict, ):
        """
        Generates a clipped parameter grid for GridSearchCV.

        Args:
            best_params: Trained best_params dict.

        Returns:
            Clipped parameter grid dictionary.
        """
        param_grid = get_base_param_grid(best_params, model_config.hyperparameter_set)
        return model_config.clip_param_grid(param_grid)

# Static Data Processing

In [3]:
def parse_size_string(size_str):
    """
    Convert size strings like "64M" or "128M" to numeric values in bytes.
    """
    if isinstance(size_str, str):
        if size_str.endswith("K"):
            return int(size_str[:-1]) * 1024
        elif size_str.endswith("M"):
            return int(size_str[:-1]) * 1024 * 1024
        elif size_str.endswith("G"):
            
            return int(size_str[:-1]) * 1024 * 1024 * 1024
        else:
            return int(size_str)  # Assume bytes if no unit is specified
    return size_str  # Return as-is if already numeric


data = pd.read_parquet(RAW_DATA_PATH)
data["write_buffer_size"] = data["write_buffer_size"].apply(parse_size_string)
data = data[FEATURE_VARS + TARGET_VARS]
print(data)
data.to_parquet(PROCESSED_DATA_PATH, index=False)



KeyboardInterrupt: 

# Dynamic Dataset production

In [ ]:
# Define data-based constants
OPERATION_TYPES = pd.read_parquet(PROCESSED_DATA_PATH)['operation_type'].unique()


def get_dataframe(model_config: ModelConfig, op_type: str):
    """
    Retrieves and processes a DataFrame based on the ModelConfig and operation type.

    Args:
        model_config: ModelConfig object.
        op_type: Operation type (GET, PUT, SEEK, OVERALL)

    Returns:
        Processed DataFrame.
    """
    feature_vars = model_config.feature_vars.copy()
    

    data = pd.read_parquet(PROCESSED_DATA_PATH) #changed to parquet

    if op_type in OPERATION_TYPES.tolist():
        data = data[data["operation_type"] == op_type]
        del data['operation_type']
        feature_vars.remove('operation_type')

    outlier_groupby_vars = feature_vars.copy()
    outlier_groupby_vars.remove('number_of_operations') # data in certain stratum maybe be bad, so IQR removal within stratum may not be reliable.
    
    def remove_outliers_iqr(group):
        """Removes outliers using the IQR method."""
        q1 = group["latency"].quantile(0.25)
        q3 = group["latency"].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        return group[(group["latency"] >= lower_bound) & (group["latency"] <= upper_bound)]

    def remove_outliers_robust_zscore(group):
        """Removes outliers using robust Z-score (MAD)."""
        median = group["latency"].median()
        mad = np.median(np.abs(group["latency"] - median))
        modified_z_scores = 0.6745 * (group["latency"] - median) / mad
        return group[np.abs(modified_z_scores) <= 3.5] #3.5 is more robust than 3.

    def winsorize_latency(group):
        """Winsorizes latency to limit extreme values."""
        lower_bound = group["latency"].quantile(0.01)  # 1st percentile
        upper_bound = group["latency"].quantile(0.99)  # 99th percentile
        group["latency"] = np.clip(group["latency"], lower_bound, upper_bound)
        return group
    def remove_outliers(group):
        scaler = StandardScaler()
        scaled_latency = scaler.fit_transform(group[["latency"]])
        z_scores = (scaled_latency - scaled_latency.mean()) / scaled_latency.std()
        return group[np.abs(z_scores) <= 3]
        
    prev_rows = data.shape[0]
    if model_config.outlier_removal_strategy == OUTLIER_REMOVAL_STRATEGY_IQR:
        data = data.groupby(outlier_groupby_vars)[feature_vars + TARGET_VARS].apply(remove_outliers_iqr).reset_index(drop=True)
        print(f"[outlier remova] IQR: {prev_rows} -> {data.shape[0]} rows")
    elif model_config.outlier_removal_strategy == OUTLIER_REMOVAL_STRATEGY_ROBUST_Z_SCORE:
        data = data.groupby(outlier_groupby_vars)[feature_vars + TARGET_VARS].apply(remove_outliers_robust_zscore).reset_index(drop=True)
        print(f"[outlier remova] ROBUST Z_SCORE: {prev_rows} -> {data.shape[0]} rows")
    elif model_config.outlier_removal_strategy == OUTLIER_REMOVAL_STRATEGY_Z_SCORE:
        data = data.groupby(outlier_groupby_vars)[feature_vars + TARGET_VARS].apply(remove_outliers).reset_index(drop=True)
    elif model_config.outlier_removal_strategy == OUTLIER_REMOVAL_STRATEGY_Z_SCORE_WINSORIZE:
        data = data.groupby(outlier_groupby_vars)[feature_vars + TARGET_VARS].apply(winsorize_latency).reset_index(drop=True)
        print(f"[outlier remova] WINSORIZE: {prev_rows} -> {data.shape[0]} rows")
    
    if model_config.hyperparameter_speedup_strategy == HYPERPARAMETER_SPEEDUP_BY_SAMPLING:
        def sample(group):
            return group.sample(frac=HYPERPARAMETER_SAMPLE_FRAC, random_state=42)
        data = data.groupby(outlier_groupby_vars)[feature_vars + ["latency"]].apply(sample).reset_index(drop=True)

    data = dynamic_aggregate(model_config, data, feature_vars)
    print(f"[get_dataframe] rows: {data.shape[0]}, columns: {data.columns}")
    # data.to_parquet(f"./debug/{op_type}_{model_config.clean_data}.parquet", index=False)
    return data, feature_vars

def dynamic_aggregate(model_config: ModelConfig, data, feature_vars: list[str], ):
    """
    Dynamically aggregates data based on the ModelConfig's aggregation type.

    Args:
        model_config: ModelConfig object.
        data: DataFrame to aggregate.
        feature_vars: List of feature variable names.

    Returns:
        Aggregated DataFrame.
    """
    agg = model_config.aggregate
    if agg == AGGREGATE_NONE:
        return data
    elif agg == AGGREGATE_MEAN:
        return data.groupby(feature_vars).agg({'latency': 'mean'}).reset_index()
    elif agg == AGGREGATE_MEDIAN:
        return data.groupby(feature_vars).agg({'latency': 'median'}).reset_index()
    else:
        raise Exception(f"Unsupported aggregation type {agg}")

def get_dynamic_dataset(model_config, op_type):
    """
    Retrieves and processes a dynamic dataset based on the ModelConfig and operation type.

    Args:
        model_config: ModelConfig object.
        op_type: Operation type.

    Returns:
        Tuple containing the DataFrame, features (X), and target (y).
    """
    df, feature_vars = get_dataframe(model_config, op_type)
    categorical_vars = CATEGORICAL_VARIABLES.copy()
    if op_type != "OVERALL":
        categorical_vars.remove('operation_type')
    categorical_vars = list(categorical_vars)

    if model_config.encoding_type == DUMMY_ENCODING:
        df = pd.get_dummies(df, columns=categorical_vars, drop_first=True)
        x = df.drop("latency", axis=1)
    elif model_config.encoding_type == ONE_HOT_ENCODING:
        ct = ColumnTransformer([('encoder', OneHotEncoder(drop='first'), categorical_vars)], remainder='passthrough')
        x = ct.fit_transform(df.drop("latency", axis=1))
    y = df["latency"]

    return df, x, y, feature_vars

[outlier remova] IQR: 16383500 -> 14509572 rows
[get_dataframe] rows: 14509572, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')


/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(full_path, bbox_inches='tight')
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(full_path, bbox_inches='tight')
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


Generated 35 plots and CSV report in ./plots/GET_feature_groups
[outlier remova] IQR: 16383500 -> 14288160 rows
[get_dataframe] rows: 14288160, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')


/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(full_path, bbox_inches='tight')
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(full_path, bbox_inches='tight')
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


Generated 35 plots and CSV report in ./plots/PUT_feature_groups
[outlier remova] IQR: 16383500 -> 14926010 rows
[get_dataframe] rows: 14926010, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')


/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(full_path, bbox_inches='tight')
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(full_path, bbox_inches='tight')
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


Generated 35 plots and CSV report in ./plots/SEEK_feature_groups
[outlier remova] IQR: 49150500 -> 43723742 rows
[get_dataframe] rows: 43723742, columns: Index(['operation_type', 'data_size', 'number_of_operations', 'latency'], dtype='object')


/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(full_path, bbox_inches='tight')
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(full_path, bbox_inches='tight')
/var/folders/3j/s7l228111077dtl7dhz8x4cm0000gn/T/ipykernel_14064/2511529951.py:216: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


Generated 105 plots and CSV report in ./plots/OVERALL_feature_groups


In [ ]:
data = pd.read_parquet(PROCESSED_DATA_PATH) #changed to parquet
gb = data.groupby(FEATURE_VARS)[FEATURE_VARS + TARGET_VARS]
print(gb.count())

                                                 operation_type  data_size  \
operation_type data_size   number_of_operations                              
GET            104857600   100                              500        500   
                           800                             4000       4000   
                           6400                           32000      32000   
                           51200                         256000     256000   
                           409600                       2048000    2048000   
...                                                         ...        ...   
SEEK           10737418240 100                              500        500   
                           800                             4000       4000   
                           6400                           32000      32000   
                           51200                         256000     256000   
                           409600                       2048000 

# Model Training logic 

In [ ]:
model_map = ModelMap()

def _train(model_config: ModelConfig, model_map: ModelMap, op_type: str):
    """
    Trains and evaluates a machine learning model on the original, non-aggregated data.
    """
    print(f"--- Training and evaluating for operation type: {op_type} ---")

    # Get training data (aggregated or non-aggregated based on model_config)
    df_train, x_train, y_train, train_feature_vars = get_dynamic_dataset(model_config, op_type)

    # Get original, non-aggregated evaluation data
    df_eval, x_eval, y_eval, eval_feature_vars= get_dynamic_dataset(ModelConfig(
        name=model_config.name,
        encoding_type=model_config.encoding_type,
        outlier_removal_strategy=model_config.outlier_removal_strategy,
        feature_vars=model_config.feature_vars,
        target_vars=model_config.target_vars,
        aggregate=AGGREGATE_NONE, # force it to be none.
    ), op_type)

    X_train, X_test, y_train, y_test = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

    scaler_x = StandardScaler()
    X_train_scaled = scaler_x.fit_transform(X_train)
    X_test_scaled = scaler_x.transform(X_test)

    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    y_test_op_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

    model = None
    if model_config.name == 'mlp':
        model = MLPRegressor(random_state=42, early_stopping=True, max_iter=200)
    elif model_config.name == 'xgb':
        model = XGBRegressor(random_state=42)

    # Hyperparameter tuning
    if model_config.hyperparameter_speedup_strategy != HYPERPARAMETER_SPEEDUP_USE_AGG:
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=model_config.param_distributions,
            n_iter=3000,
            scoring='neg_mean_squared_error',
            cv=5,
            n_jobs=-1,
            random_state=42
        )
        random_search.fit(X_train_scaled, y_train_scaled)

        # Narrow down ranges for GridSearchCV
        print(f"Best parameters from RandomSearchCV: {random_search.best_params_}")
        param_grid = get_param_grid(model_config, random_search.best_params_)
        print(f"Clipped param grid from RandomSearchCV: {param_grid}")
    else:
        param_grid = model_map.getModelResult(op_type, model_config)

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='neg_mean_squared_error',
        cv=5,
        n_jobs=-1,
    )
    grid_search.fit(X_train_scaled, y_train_scaled)

    best_model_grid = grid_search.best_estimator_

    # Evaluation on original, non-aggregated data
    X_eval_scaled = scaler_x.transform(x_eval) #Scale the evaluation data using the training scaler.
    y_eval_scaled = scaler_y.transform(y_eval.values.reshape(-1, 1)).flatten()

    y_pred_scaled = best_model_grid.predict(X_eval_scaled)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    mse = mean_squared_error(y_eval, y_pred)
    r2 = r2_score(y_eval, y_pred)

    if op_type == "OVERALL":
        model_map.addOverallModelResult(model_config, ModelResult(best_model_grid, mse, r2, get_param_grid(model_config, grid_search.best_params_)))
    else:
        model_map.addIndivModelResult(op_type, model_config, ModelResult(best_model_grid, mse, r2, get_param_grid(model_config, grid_search.best_params_)))

    print(f"Best parameters from GridSearchCV: {grid_search.best_params_}")
    print(f"Mean Squared Error ({op_type}): {mse}")
    print(f"R-squared ({op_type}): {r2}")
    print("-" * 50)

def train_models(model_config: ModelConfig):
    print(f"#"*10, str(model_config), '#'*10)
    for op_type in OPERATION_TYPES:
        _train(model_config, model_map, op_type)
    _train(model_config, model_map, "OVERALL")

In [6]:
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = ('ignore::UserWarning,ignore::ConvergenceWarning,ignore::RuntimeWarning')

In [6]:
train_models(
  ModelConfig(
    'mlp', ONE_HOT_ENCODING, OUTLIER_REMOVAL_STRATEGY_IQR, FEATURE_VARS, TARGET_VARS, AGGREGATE_MEAN
  )
)

########## ModelConfig(name=mlp, encoding_type=one_hot, outlier_removal_strategy=OUTLIER_REMOVAL_STRATEGY_IQR, feature_vars=['operation_type', 'data_size', 'number_of_operations'], target_vars=['latency'], agg=AGGREGATE_MEAN), hyper_param_speedup=None ##########
--- Training and evaluating for operation type: GET ---
[outlier remova] IQR: 16383500 -> 14509572 rows
[get_dataframe] rows: 35, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')
[outlier remova] IQR: 16383500 -> 14509572 rows
[get_dataframe] rows: 14509572, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')


Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'


Best parameters from RandomSearchCV: {'activation': 'tanh', 'alpha': np.float64(0.0023002015982368533), 'hidden_layer_sizes': (63,), 'learning_rate': 'adaptive', 'learning_rate_init': np.float64(0.042222824844050114), 'solver': 'lbfgs'}
Clipped param grid from RandomSearchCV: {'alpha': array([0.0001   , 0.0001   , 0.0023002, 0.0048002, 0.0073002]), 'learning_rate_init': array([0.03222282, 0.03722282, 0.04222282, 0.04722282, 0.05222282]), 'hidden_layer_sizes': [(63,)], 'activation': ['tanh'], 'learning_rate': ['adaptive'], 'solver': ['lbfgs']}
Added Indiv Result
ModelMap:

Best parameters from GridSearchCV: {'activation': 'tanh', 'alpha': np.float64(0.0023002015982368533), 'hidden_layer_sizes': (63,), 'learning_rate': 'adaptive', 'learning_rate_init': np.float64(0.03222282484405011), 'solver': 'lbfgs'}
Mean Squared Error (GET): 2008.2867956444816
R-squared (GET): 0.3785540235982844
--------------------------------------------------
--- Training and evaluating for operation type: PUT ---

In [7]:
train_models(
  ModelConfig(
    'mlp', ONE_HOT_ENCODING, OUTLIER_REMOVAL_STRATEGY_IQR, FEATURE_VARS, TARGET_VARS, AGGREGATE_MEDIAN
  )
)

########## ModelConfig(name=mlp, encoding_type=one_hot, outlier_removal_strategy=OUTLIER_REMOVAL_STRATEGY_IQR, feature_vars=['operation_type', 'data_size', 'number_of_operations'], target_vars=['latency'], agg=AGGREGATE_MEDIAN), hyper_param_speedup=None ##########
--- Training and evaluating for operation type: GET ---
[outlier remova] IQR: 16383500 -> 14509572 rows
[get_dataframe] rows: 35, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')
[outlier remova] IQR: 16383500 -> 14509572 rows
[get_dataframe] rows: 14509572, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')


Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: Invalid -W option ignored:unknown warning category: 'ConvergenceWarning' 
unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'


Best parameters from RandomSearchCV: {'activation': 'relu', 'alpha': np.float64(0.00168705360610974), 'hidden_layer_sizes': (19, 79), 'learning_rate': 'constant', 'learning_rate_init': np.float64(0.053907926869519784), 'solver': 'lbfgs'}
Clipped param grid from RandomSearchCV: {'alpha': array([0.0001    , 0.0001    , 0.00168705, 0.00418705, 0.00668705]), 'learning_rate_init': array([0.04390793, 0.04890793, 0.05390793, 0.05890793, 0.06390793]), 'hidden_layer_sizes': [(19, 79)], 'activation': ['relu'], 'learning_rate': ['constant'], 'solver': ['lbfgs']}
Added Indiv Result
ModelMap:

=== Model: mlp ===
+-----------+------------------------------+-----------------+----------------+-------------+------------+
| Op Type   | Clean Data                   | Encoding Type   | Aggregate      |         MSE |         R² |
+===========+==============================+=================+================+=============+============+
| GET       | OUTLIER_REMOVAL_STRATEGY_IQR | one_hot         | AGGREGATE

In [48]:
train_models(
  ModelConfig(
    'xgb', ONE_HOT_ENCODING, OUTLIER_REMOVAL_STRATEGY_IQR, FEATURE_VARS, TARGET_VARS, AGGREGATE_MEAN
  )
)

########## ModelConfig(name=xgb, encoding_type=one_hot, outlier_removal_strategy=all, feature_vars=['operation_type', 'data_size', 'number_of_operations'], target_vars=['latency'], agg=AGGREGATE_MEAN), hyper_param_speedup=None ##########
--- Training and evaluating for operation type: GET ---
[outlier remova] ROBUST Z_SCORE: 16383500 -> 10924941 rows
[get_dataframe] rows: 31, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')
[outlier remova] ROBUST Z_SCORE: 16383500 -> 10924941 rows
[get_dataframe] rows: 10924941, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')
Best parameters from RandomSearchCV: {'colsample_bytree': np.float64(0.7980679789166819), 'learning_rate': np.float64(0.012051694554007274), 'max_depth': 7, 'n_estimators': 191, 'subsample': np.float64(0.7339809259194057)}
Clipped param grid from RandomSearchCV: {'colsample_bytree': array([0.78806798, 0.79306798, 0.79806798, 0.80306798, 0.80806798]), 'learning_rate'

In [ ]:
train_models(
  ModelConfig(
    'xgb', ONE_HOT_ENCODING, OUTLIER_REMOVAL_STRATEGY_IQR, FEATURE_VARS, TARGET_VARS, AGGREGATE_MEDIAN
  )
)

In [8]:
train_models(
  ModelConfig(
    'mlp', ONE_HOT_ENCODING, OUTLIER_REMOVAL_STRATEGY_IQR, FEATURE_VARS, TARGET_VARS, AGGREGATE_NONE, HYPERPARAMETER_SPEEDUP_BY_SAMPLING
  )
)

########## ModelConfig(name=mlp, encoding_type=one_hot, outlier_removal_strategy=OUTLIER_REMOVAL_STRATEGY_IQR, feature_vars=['operation_type', 'data_size', 'number_of_operations'], target_vars=['latency'], agg=AGGREGATE_NONE), hyper_param_speedup=HYPERPARAMETER_SPEEDUP_BY_SAMPLING ##########
--- Training and evaluating for operation type: GET ---
[outlier remova] IQR: 16383500 -> 14509572 rows
[get_dataframe] rows: 14507, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')
[outlier remova] IQR: 16383500 -> 14509572 rows
[get_dataframe] rows: 14509572, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')


Invalid -W option ignored:Invalid -W option ignored:  unknown warning category: 'ConvergenceWarning'unknown warning category: 'ConvergenceWarning'

Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'


Best parameters from RandomSearchCV: {'activation': 'relu', 'alpha': np.float64(0.04786412847960981), 'hidden_layer_sizes': (66,), 'learning_rate': 'invscaling', 'learning_rate_init': np.float64(0.06557523597900597), 'solver': 'lbfgs'}
Clipped param grid from RandomSearchCV: {'alpha': array([0.03786413, 0.04286413, 0.04786413, 0.05      , 0.05      ]), 'learning_rate_init': array([0.05557524, 0.06057524, 0.06557524, 0.07057524, 0.07557524]), 'hidden_layer_sizes': [(66,)], 'activation': ['relu'], 'learning_rate': ['invscaling'], 'solver': ['lbfgs']}
Added Indiv Result
ModelMap:

=== Model: mlp ===
+-----------+------------------------------+-----------------+------------------+-------------+--------------+
| Op Type   | Clean Data                   | Encoding Type   | Aggregate        |         MSE |           R² |
+===========+==============================+=================+==================+=============+==============+
| GET       | OUTLIER_REMOVAL_STRATEGY_IQR | one_hot         | 

Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'
Invalid -W option ignored: unknown warning category: 'ConvergenceWarning'


Best parameters from RandomSearchCV: {'activation': 'relu', 'alpha': np.float64(0.01281290255779814), 'hidden_layer_sizes': (137, 57), 'learning_rate': 'adaptive', 'learning_rate_init': np.float64(0.012192243815616503), 'solver': 'lbfgs'}
Clipped param grid from RandomSearchCV: {'alpha': array([0.0028129, 0.0078129, 0.0128129, 0.0178129, 0.0228129]), 'learning_rate_init': array([0.00219224, 0.00719224, 0.01219224, 0.01719224, 0.02219224]), 'hidden_layer_sizes': [(137, 57)], 'activation': ['relu'], 'learning_rate': ['adaptive'], 'solver': ['lbfgs']}


: 

In [ ]:
def grid_search_only(model_config: ModelConfig, model_map: ModelMap, op_type: str, clipped_param_grid: dict):
    print(f"--- Training and evaluating for operation type: {op_type} ---")

    # Get training data (aggregated or non-aggregated based on model_config)
    df_train, x_train, y_train, feature_vars = get_dynamic_dataset(model_config, op_type)

    # Get original, non-aggregated evaluation data
    df_eval, x_eval, y_eval = get_dynamic_dataset(ModelConfig(
        name=model_config.name,
        encoding_type=model_config.encoding_type,
        outlier_removal_strategy=model_config.outlier_removal_strategy,
        feature_vars=model_config.feature_vars,
        target_vars=model_config.target_vars,
        aggregate=AGGREGATE_NONE, # force it to be none.
    ), op_type)

    X_train, X_test, y_train, y_test = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

    scaler_x = StandardScaler()
    X_train_scaled = scaler_x.fit_transform(X_train)
    X_test_scaled = scaler_x.transform(X_test)

    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    y_test_op_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

    model = None
    if model_config.name == 'mlp':
        model = MLPRegressor(random_state=42, early_stopping=True, max_iter=200, solver='adam')
    elif model_config.name == 'xgb':
        model = XGBRegressor(random_state=42)

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=clipped_param_grid,
        scoring='neg_mean_squared_error',
        cv=5,
        n_jobs=2,
    )
    grid_search.fit(X_train_scaled, y_train_scaled)

    best_model_grid = grid_search.best_estimator_

    # Evaluation on original, non-aggregated data
    X_eval_scaled = scaler_x.transform(x_eval) #Scale the evaluation data using the training scaler.
    y_eval_scaled = scaler_y.transform(y_eval.values.reshape(-1, 1)).flatten()

    y_pred_scaled = best_model_grid.predict(X_eval_scaled)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    mse = mean_squared_error(y_eval, y_pred)
    r2 = r2_score(y_eval, y_pred)

    if op_type == "OVERALL":
        model_map.addOverallModelResult(model_config, ModelResult(best_model_grid, mse, r2, get_param_grid(model_config, grid_search.best_params_)))
    else:
        model_map.addIndivModelResult(op_type, model_config, ModelResult(best_model_grid, mse, r2, get_param_grid(model_config, grid_search.best_params_)))

    print(f"Best parameters from GridSearchCV: {grid_search.best_params_}")
    print(f"Mean Squared Error ({op_type}): {mse}")
    print(f"R-squared ({op_type}): {r2}")
    print("-" * 50)

clipped_param_grid = {'alpha': [0.00345164, 0.00595164, 0.00845164, 0.01095164, 0.01345164], 'learning_rate_init': [0.00158563, 0.00658563, 0.01158563, 0.01658563, 0.02158563], 'hidden_layer_sizes': [(137, 57)], 'activation': ['tanh'], 'learning_rate': ['adaptive'], 'solver': ['adam']}
grid_search_only(ModelConfig(
    'mlp', ONE_HOT_ENCODING, OUTLIER_REMOVAL_STRATEGY_IQR, FEATURE_VARS, TARGET_VARS, AGGREGATE_NONE, HYPERPARAMETER_SPEEDUP_BY_SAMPLING
  ), model_map, "OVERALL", clipped_param_grid)

--- Training and evaluating for operation type: OVERALL ---
[outlier remova] IQR: 49150500 -> 43723742 rows
[get_dataframe] rows: 43721, columns: Index(['operation_type', 'data_size', 'number_of_operations', 'latency'], dtype='object')
[outlier remova] IQR: 49150500 -> 43723742 rows
[get_dataframe] rows: 43723742, columns: Index(['operation_type', 'data_size', 'number_of_operations', 'latency'], dtype='object')


: 

In [ ]:
train_models(
  ModelConfig(
    'xgb', ONE_HOT_ENCODING, True, FEATURE_VARS, TARGET_VARS, AGGREGATE_NONE, HYPERPARAMETER_SPEEDUP_BY_SAMPLING
  )
)

In [27]:
# sample frac=0.001
train_models(
  ModelConfig(
    'xgb', ONE_HOT_ENCODING, True, FEATURE_VARS, TARGET_VARS, AGGREGATE_NONE, HYPERPARAMETER_SPEEDUP_BY_SAMPLING
  )
)

########## ModelConfig(name=xgb, encoding_type=one_hot, clean_data=True, feature_vars=['operation_type', 'data_size', 'number_of_operations'], target_vars=['latency'], agg=AGGREGATE_NONE) ##########
--- Training and evaluating for operation type: GET ---
[get_dataframe] rows: 161985, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')
Best parameters from RandomSearchCV: {'colsample_bytree': np.float64(0.9711171901525811), 'learning_rate': np.float64(0.19794513286453821), 'max_depth': 9, 'n_estimators': 151, 'subsample': np.float64(0.7194959228448682)}
Clipped param grid from RandomSearchCV: {'subsample': array([0.70949592, 0.71449592, 0.71949592, 0.72449592, 0.72949592]), 'max_depth': [9], 'colsample_bytree': array([0.96111719, 0.96611719, 0.97111719, 0.97611719, 0.98111719]), 'n_estimators': [151], 'learning_rate': array([0.18794513, 0.19294513, 0.19794513, 0.2       , 0.2       ])}
Added Indiv Result
ModelMap:

Best parameters from GridSearchCV: {'colsam

In [ ]:
def save_model(model_config: ModelConfig, model_result: ModelResult, op_type: str):
    """
    Saves a trained model with a DateTime prefix to the filename.
    """
    name, agg, model = model_config.name, model_config.aggregate, model_result.model
    hyp_strat = 'none'
    if model_config.hyperparameter_speedup_strategy:
        hyp_strat = model_config.hyperparameter_speedup_strategy

    # Get current DateTime and format it
    now = datetime.now()
    datetime_prefix = now.strftime("%Y%m%d_%H%M%S_")

    file_name = f'./models/{datetime_prefix}{op_type}-{name}-{agg}-{hyp_strat}'

    if name == 'mlp':
        joblib.dump(model, f'{file_name}.joblib')
    elif name == 'xgb':
        model.save_model(f'{file_name}.json')

def load_model(model_config: ModelConfig, op_type: str):
    """
    Loads a trained model from the filesystem.

    Args:
        model_config: The ModelConfig object used to train the model.
        op_type: The operation type associated with the model.

    Returns:
        The loaded model, or None if the model file is not found.
    """
    name, agg = model_config.name, model_config.aggregate
    hyp_strat = 'none'
    if model_config.hyperparameter_speedup_strategy:
        hyp_strat = model_config.hyperparameter_speedup_strategy
    file_name = f'./models/{op_type}-{name}-{agg}-{hyp_strat}'

    if model_config.name == 'mlp':
        file_path = f'{file_name}.joblib'
        if os.path.exists(file_path):
            return joblib.load(file_path)
        else:
            print(f"Model file not found: {file_path}")
            return None
    elif model_config.name == 'xgb':
        file_path = f'{file_name}.json'
        if os.path.exists(file_path):
            loaded_model = XGBRegressor()
            loaded_model.load_model(file_path)
            return loaded_model
        else:
            print(f"Model file not found: {file_path}")
            return None
    else:
        print(f"Unsupported model type: {model_config.name}")
        return None

def evaluate_model(aggregated_model, model_config, op_type):
    """
    Tests a model trained on aggregated data against non-aggregated data.

    Args:
        aggregated_model: The trained model from aggregated data.
        model_config: The ModelConfig object.
        op_type: The operation type to test.
    """
    print(f"--- Evaluating Model on Non-Aggregated Data ({op_type}) ---")
    print(model_config)

    # 1. Get Non-Aggregated Data
    original_agg, original_hyp = model_config.aggregate, model_config.hyperparameter_speedup_strategy
    model_config.aggregate, model_config.hyperparameter_speedup_strategy = AGGREGATE_NONE, None
    non_aggregated_df, feature_vars = get_dataframe(model_config, op_type)
    model_config.aggregate, model_config.hyperparameter_speedup_strategy = original_agg, original_hyp
    

    # 2. Prepare Data for Model
    categorical_vars = CATEGORICAL_VARIABLES.copy()
    if op_type != "OVERALL":
        categorical_vars.remove('operation_type')
    categorical_vars = list(categorical_vars)

    if model_config.encoding_type == "dummy":
        non_aggregated_df = pd.get_dummies(non_aggregated_df, columns=categorical_vars, drop_first=True)
        x = non_aggregated_df.drop("latency", axis=1)
    elif model_config.encoding_type == "one_hot":
        ct = ColumnTransformer([('encoder', OneHotEncoder(drop='first'), categorical_vars)], remainder='passthrough')
        x = ct.fit_transform(non_aggregated_df.drop("latency", axis=1))

    y = non_aggregated_df["latency"]

    # 3. Split Data
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    # 4. Scale Data
    scaler_x = StandardScaler()
    X_test_scaled = scaler_x.fit_transform(X_test)
    scaler_y = StandardScaler()
    y_test_scaled = scaler_y.fit_transform(y_test.values.reshape(-1, 1)).flatten()

    # 5. Make Predictions
    y_pred_scaled = aggregated_model.predict(X_test_scaled)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    # 6. Evaluate Model
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Mean Squared Error: {mse}")
    print(f"R-squared: {r2}")
    print("-" * 50)

    return mse, r2
    
def human_readable_size(size_bytes, for_filename=False):
    """Convert bytes to human-readable format matching original C++ values"""
    # Original C++ size mappings
    size_mappings = {
        104857600: '100MB',
        209715200: '200MB',
        524288000: '500MB',
        1073741824: '1GB',
        2147483648: '2GB',
        5368709120: '5GB',
        10737418240: '10GB'
    }
    
    # First try exact matches from original C++ values
    if size_bytes in size_mappings:
        base_name = size_mappings[size_bytes]
        return base_name.lower() if for_filename else base_name
    
    # Fallback for other values (shouldn't occur with original data)
    size_names = ['bytes', 'KB', 'MB', 'GB', 'TB']
    unit_index = min(int(math.log2(size_bytes) // 10), len(size_names) - 1)
    size = size_bytes / (1024 ** unit_index)
    
    if for_filename:
        return f"{int(size)}{size_names[unit_index].lower()}"
    return f"{size:.1f} {size_names[unit_index]}"

def clean_filename(text):
    """Convert text to clean filename format"""
    # Handle number-unit combinations (e.g., "1.5 MB" → "15mb")
    text = re.sub(r'(\d+)\.(\d+)\s*([a-z]{2})', 
                 lambda m: f"{m.group(1)}{m.group(2)}{m.group(3)}", 
                 text.lower())
    # Replace other special characters
    return re.sub(r'[^\w]+', '_', text).strip('_')

def generate_plot_filename(row, op_type):
    """Generate filename in format: <MSE>_<OPS>_<SIZE>.png or <MSE>_<OPTYPE>_<OPS>_<SIZE>.png"""
    # Extract values from row
    mse = row['mse']
    num_ops = row['number_of_operations']
    data_size = row['data_size']
    
    # Get size string (e.g., "10gb")
    size_str = human_readable_size(data_size, for_filename=True)
    
    # Format filename based on operation type
    if op_type == "OVERALL":
        # For overall, include operation type from display_name (e.g., "data_size=10GB, ops=100")
        op_match = re.search(r'operation_type=(\w+)', row['display_name'])
        if op_match:
            op_str = op_match.group(1).lower()
            return f"{mse:.0f}_{op_str}_{num_ops}_{size_str}.png"
    
    return f"{mse:.0f}_{num_ops}_{size_str}.png"
    

def evaluate_by_feature_groups_with_plots(model_config: ModelConfig, op_type: str):
    """
    Evaluates model performance by feature combinations and saves diagnostic plots
    
    Args:
        model_config: Model configuration used for training
        op_type: Operation type being evaluated
    """
    # Load the trained model
    model = load_model(model_config, op_type)
    if model is None:
        print(f"No trained model found for {op_type}")
        return
    
    # Load evaluation data (non-aggregated)
    df_eval, x_eval, y_eval, feature_vars = get_dynamic_dataset(
        ModelConfig(
            name=model_config.name,
            encoding_type=model_config.encoding_type,
            outlier_removal_strategy=model_config.outlier_removal_strategy,
            feature_vars=model_config.feature_vars,
            target_vars=model_config.target_vars,
            aggregate=AGGREGATE_NONE
        ),
        op_type
    )
    
    # Add predictions
    df_eval['predicted'] = model.predict(x_eval)
    
    # Create human-readable data_size column for display
    if 'data_size' in df_eval.columns:
        df_eval['data_size_display'] = df_eval['data_size'].apply(human_readable_size)
    
    # Calculate MSE per feature combination
    results = []
    for group_vals, group_data in df_eval.groupby(feature_vars):
        # Create display-friendly versions of all values
        display_parts = []
        file_parts = []
        csv_parts = []
        
        if len(feature_vars) > 1:
            for col, val in zip(feature_vars, group_vals):
                if col == 'data_size':
                    disp_val = human_readable_size(val)
                    file_val = disp_val.lower().replace(' ', '')
                    csv_val = disp_val
                else:
                    disp_val = str(val)
                    file_val = disp_val
                    csv_val = disp_val
                
                display_parts.append(f"{col}={disp_val}")
                file_parts.append(f"{file_val}")
                csv_parts.append(csv_val)
                
            display_name = ", ".join(display_parts)
            file_name = "_".join(file_parts)
            csv_name = "_".join(csv_parts)
        else:
            group_name = str(group_vals)
            display_name = f"{feature_vars[0]}={human_readable_size(group_vals) if feature_vars[0] == 'data_size' else group_vals}"
        
        mse = mean_squared_error(group_data['latency'], group_data['predicted'])
        
        results.append({
            'group': csv_name,  # For CSV (e.g., "10gb_100")
            'display_name': display_name,  # For plots (e.g., "data_size=10GB, ops=100")
            'mse': mse,
            'count': len(group_data),
            'latency_mean': group_data['latency'].mean(),
            'latency_std': group_data['latency'].std(),
            **dict(zip(
                [f"{col}_display" if col == 'data_size' else col 
                 for col in feature_vars],
                [human_readable_size(v) if col == 'data_size' else v 
                 for col, v in zip(feature_vars, group_vals if len(feature_vars) > 1 else [group_vals])]
            )),
            **dict(zip(feature_vars, group_vals if len(feature_vars) > 1 else [group_vals]))
        })
    
    results_df = pd.DataFrame(results).sort_values('mse', ascending=False)
    
    # Create directory for plots
    plot_dir = f"./plots/{op_type}_feature_groups"
    os.makedirs(plot_dir, exist_ok=True)
    
    # Generate plots for ALL groups (not limited to top N)
    for _, row in results_df.iterrows():
        # Reconstruct group query using original values
        group_query = ' & '.join([
            f"{col} == {repr(row[col])}" 
            for col in feature_vars
        ])
        group_data = df_eval.query(group_query)
        
        plt.figure(figsize=(12, 6))
        
        # Actual vs Predicted plot with red ideal line
        plt.subplot(1, 2, 1)
        plt.scatter(group_data['latency'], group_data['predicted'], alpha=0.5)
        # The red dashed line represents perfect predictions (where actual = predicted)
        plt.plot([group_data['latency'].min(), group_data['latency'].max()], 
                [group_data['latency'].min(), group_data['latency'].max()], 
                'r--', label='Perfect prediction')
        plt.xlabel('Actual Latency')
        plt.ylabel('Predicted Latency')
        plt.title(f'Actual vs Predicted\n{row["display_name"]}')
        plt.legend()
        
        # Latency distribution plot
        plt.subplot(1, 2, 2)
        plt.hist(group_data['latency'], bins=30, alpha=0.7)
        plt.xlabel('Latency')
        plt.ylabel('Frequency')
        plt.title(f'Latency Distribution\nMSE: {row["mse"]:.2f} (n={row["count"]})')

        plot_filename = generate_plot_filename(row, op_type)
        full_path = f"{plot_dir}/{plot_filename}"
        
        plt.tight_layout()
        plt.savefig(full_path, bbox_inches='tight')
        plt.close()
    
    # Save detailed results with human-readable values
    results_df.to_csv(
        f"{plot_dir}/{op_type}_feature_group_performance.csv", 
        index=False,
        float_format="%.2f"
    )
    
    print(f"Generated {len(results_df)} plots and CSV report in {plot_dir}")
    return results_df

for op_type in OPERATION_TYPES.tolist() +['OVERALL']:
    evaluate_by_feature_groups_with_plots(ModelConfig(
    'xgb', ONE_HOT_ENCODING, OUTLIER_REMOVAL_STRATEGY_IQR, FEATURE_VARS, TARGET_VARS, AGGREGATE_NONE, HYPERPARAMETER_SPEEDUP_BY_SAMPLING
  ), op_type)

In [ ]:
for op_type, obj in model_map.data.items():
    for model_name, tuples in obj.items():
        for tup in tuples:
            model_config, model_result = tup
            save_model(model_config, model_result, op_type)
            
for model_name, tuples in model_map.overall_results.items():
    for tup in tuples:
        model_config, model_result = tup
        save_model(model_config, model_result, "OVERALL")

In [6]:
for filename in os.listdir(MODEL_DIR):
        if filename.endswith(".joblib") or filename.endswith(".json"):
            # Extract information from filename
            parts = filename.split("-")
            if len(parts) >= 4:
                op_type = parts[0]
                model_name = parts[1]
                agg = parts[2]
                hyp_strat = parts[3].split(".")[0] # remove extension

                # Create a dummy ModelConfig object (adjust as needed)
                model_config = ModelConfig(
                    name=model_name,
                    encoding_type=ONE_HOT_ENCODING,  # Adjust encoding type if necessary
                    outlier_removal_strategy=None, # adjust as needed
                    feature_vars=FEATURE_VARS, #adjust as needed
                    target_vars=TARGET_VARS, #adjust as needed
                    aggregate=agg,
                    hyperparameter_speedup_strategy=hyp_strat,
                )

                loaded_model = load_model(model_config, op_type)

                if loaded_model:
                    evaluate_model(loaded_model, model_config, op_type)
                else:
                    print(f"Failed to load model: {filename}")
            else:
                print(f"Invalid filename format: {filename}")

--- Evaluating Model on Non-Aggregated Data (GET) ---
ModelConfig(name=mlp, encoding_type=one_hot, clean_data=True, feature_vars=['operation_type', 'data_size', 'number_of_operations'], target_vars=['latency'], agg=AGGREGATE_MEAN), hyper_param_speedup=none
[get_dataframe] rows: 16198575, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')
Mean Squared Error: 50314.23954177457
R-squared: -0.4220115501997115
--------------------------------------------------
--- Evaluating Model on Non-Aggregated Data (GET) ---
ModelConfig(name=mlp, encoding_type=one_hot, clean_data=True, feature_vars=['operation_type', 'data_size', 'number_of_operations'], target_vars=['latency'], agg=AGGREGATE_MEDIAN), hyper_param_speedup=none
[get_dataframe] rows: 16198575, columns: Index(['data_size', 'number_of_operations', 'latency'], dtype='object')
Mean Squared Error: 107054.35110221409
R-squared: -2.025634992258783
--------------------------------------------------
--- Evaluating Mod